<a href="https://colab.research.google.com/github/pnperl/Travel_Router/blob/main/Travel_Router.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yfinance pandas numpy requests python-dotenv -q

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests
import time
import logging
from datetime import datetime, timedelta

from zoneinfo import ZoneInfo
from IPython.display import clear_output
from google.colab import userdata

# --- 1. SETTINGS & ASSETS ---
# Expanded list with popular symbols from different markets
SYMBOLS = [
    #"BTC-USD", "ETH-USD", "SOL-USD",  # Crypto
    "^NSEI", "RELIANCE.NS", "HDFCBANK.NS", # India (Nifty 50, Reliance, HDFC)
    #"AAPL", "TSLA", "NVDA", "MSFT"      # US (Apple, Tesla, Nvidia, Microsoft)
    #"GOLD.MCX", "SILVER.MCX", "CRUDEOIL.MCX", "COPPER.MCX", "NATURALGAS.MCX", "ZINC.MCX", "LEAD.MCX", "ALUMINIUM.MCX", "MCXBULLDEX.MCX", "MENTHAOIL.MCX", "COTTON.MCX"
]

INTERVAL = "5m"
MIN_PROBABILITY = 30 #(default 60%) → trade is skipped and counted as "Skipped" in the dashboard. Raise this to 70–75% to be even more selective.
ATR_SL_MULTIPLIER = 1.5 # try 2.0 if SL still hits too often
TP_THRESHOLD = 0.03  # 3% Fixed Take Profit
IST = ZoneInfo("Asia/Kolkata")

TOKEN = userdata.get("TELEGRAM_TOKEN")
CHAT_ID = userdata.get("CHAT_ID")

logging.basicConfig(level=logging.WARNING)

# --- 2. CORE FUNCTIONS ---
def detect_profile(symbol: str) -> dict:
    s = symbol.upper()
    if "BTC" in s or "ETH" in s or "SOL" in s or s.endswith("-USD"):
        return dict(type="CRYPTO", tz="UTC", hours=None, doji=0.15, strike=1)
    if "^NSE" in s or s.endswith(".NS"):
        return dict(type="INDIA", tz="Asia/Kolkata", hours=("09:15","15:30"), doji=0.20, strike=50)
    return dict(type="US_STOCK", tz="America/New_York", hours=("09:30","16:00"), doji=0.20, strike=1)

def is_market_open(profile: dict) -> bool:
    if profile["hours"] is None: return True
    now_tz = datetime.now(ZoneInfo(profile["tz"]))
    if now_tz.weekday() >= 5: return False
    curr = now_tz.strftime("%H:%M")
    return profile["hours"][0] <= curr <= profile["hours"][1]

def send_alert(msg):
    if not TOKEN or not CHAT_ID: print(f"[LOG] {msg}"); return
    try: requests.post(f"https://api.telegram.org/bot{TOKEN}/sendMessage", data={"chat_id": CHAT_ID, "text": msg}, timeout=5)
    except: pass

def fetch_data(symbol, interval):
    try:
        df = yf.download(symbol, period="2d", interval=interval, auto_adjust=False, progress=False)
        if isinstance(df.columns, pd.MultiIndex): df.columns = [c[0] for c in df.columns]
        return df if not df.empty else None
    except: return None

def heikin_ashi(df):
    # Select relevant columns and drop NaNs. This creates one copy.
    # Assuming yfinance already returns numeric columns, .apply(pd.to_numeric) is often redundant.
    ohlc_df = df[["Open", "High", "Low", "Close"]].dropna()

    if len(ohlc_df) < 10:
        return None

    # Extract NumPy arrays directly from the cleaned DataFrame to avoid further Pandas overhead
    open_prices = ohlc_df["Open"].values
    high_prices = ohlc_df["High"].values
    low_prices = ohlc_df["Low"].values
    close_prices = ohlc_df["Close"].values

    ha_close = (open_prices + high_prices + low_prices + close_prices) / 4

    ha_open = np.zeros_like(open_prices)
    ha_open[0] = (open_prices[0] + close_prices[0]) / 2
    for i in range(1, len(open_prices)):
        ha_open[i] = (ha_open[i-1] + ha_close[i-1]) / 2

    ha_high = np.maximum.reduce([ha_open, ha_close, high_prices])
    ha_low = np.minimum.reduce([ha_open, ha_close, low_prices])

    return pd.DataFrame({"open": ha_open, "close": ha_close, "high": ha_high, "low": ha_low})

def compute_indicators(df):
    close, high, low = df["Close"], df["High"], df["Low"]
    delta = close.diff(); gain = delta.clip(lower=0).rolling(14).mean(); loss = (-delta.clip(upper=0)).rolling(14).mean()
    rsi = float((100 - 100 / (1 + gain / loss)).iloc[-2])
    tr = pd.concat([(high - low), (high - close.shift(1)).abs(), (low - close.shift(1)).abs()], axis=1).max(axis=1)
    return {"rsi": rsi, "atr": float(tr.rolling(14).mean().iloc[-2]), "ema": float(close.ewm(span=20).mean().iloc[-2])}

def print_dashboard(symbol_states):
    clear_output(wait=True)
    print(f"\n  === MULTI-SYMBOL BOT ({datetime.now(IST).strftime('%I:%M:%S %p')}) ===")
    print(f"  {'SYMBOL':<12} {'STATUS':<8} {'POS':<6} {'ENTRY':<10} {'PRICE':<10} {'UNREAL':<10} {'TRADES':<7} {'P&L':<8} {'WINS':<5} {'LOSSES':<7} {'WIN%':<6} {'BEST':<8} {'WORST':<8}")
    print("  " + "-"*150)

    g_trades, g_pnl = 0, 0.0
    g_wins, g_losses = 0, 0
    g_best_trade = float('-inf')
    g_worst_trade = float('inf')

    for s, st in symbol_states.items():
        m_stat = "OPEN" if is_market_open(st['profile']) else "CLOSED"
        pos, ent, cur = st['position'] or "-", round(st['entry_price'] or 0, 2), round(st['latest_price'] or 0, 2)
        unreal = 0.0
        if pos == "CALL": unreal = cur - ent
        elif pos == "PUT": unreal = ent - cur

        # Symbol-specific metrics
        wins = st['wins']
        losses = st['losses']
        symbol_pnl = st['stats']['pnl']
        best_trade = round(st['best_trade'], 2) if st['best_trade'] != float('-inf') else "N/A"
        worst_trade = round(st['worst_trade'], 2) if st['worst_trade'] != float('inf') else "N/A"

        total_trades = wins + losses
        win_rate = round((wins / total_trades) * 100, 1) if total_trades > 0 else 0.0

        print(f"  {s:<12} {m_stat:<8} {pos:<6} {ent:<10} {cur:<10} {round(unreal,2):<10} {total_trades:<7} {round(symbol_pnl,2):<8} {wins:<5} {losses:<7} {win_rate:<6} {best_trade:<8} {worst_trade:<8}")

        # Update global metrics
        g_trades += total_trades # Sum of total trades for each symbol
        g_pnl += symbol_pnl
        g_wins += wins
        g_losses += losses
        if st['best_trade'] != float('-inf'): g_best_trade = max(g_best_trade, st['best_trade'])
        if st['worst_trade'] != float('inf'): g_worst_trade = min(g_worst_trade, st['worst_trade'])

    # Global Summary
    global_total_trades = g_wins + g_losses
    global_win_rate = round((g_wins / global_total_trades) * 100, 1) if global_total_trades > 0 else 0.0
    g_best_trade_display = round(g_best_trade, 2) if g_best_trade != float('-inf') else "N/A"
    g_worst_trade_display = round(g_worst_trade, 2) if g_worst_trade != float('inf') else "N/A"

    print(f"\n  Global Trades: {g_trades} | Realized P&L: {round(g_pnl, 2)} pts | Wins: {g_wins} | Losses: {g_losses} | Win Rate: {global_win_rate}% | Best Trade: {g_best_trade_display} | Worst Trade: {g_worst_trade_display}")

# --- 3. MAIN LOOP ---
def start_bot():
    states = {s: {
        "position": None,
        "entry_price": None,
        "trailing_sl": None,
        "latest_price": 0,
        "profile": detect_profile(s),
        "stats": {"trades":0, "pnl":0.0},
        "wins": 0,
        "losses": 0,
        "best_trade": float('-inf'),
        "worst_trade": float('inf'),
        "last_time": None
    } for s in SYMBOLS}

    while True:
        for s in SYMBOLS:
            st = states[s]
            if not is_market_open(st["profile"]): continue

            df = fetch_data(s, INTERVAL)
            if df is None: continue
            if df.index[-1] == st["last_time"]: continue
            st["last_time"] = df.index[-1]

            ha = heikin_ashi(df); ind = compute_indicators(df)
            price = float(df["Close"].iloc[-1]); st["latest_price"] = price

            # Exit Logic (TP & SL)
            if st["position"]:
                p_pct = (price - st["entry_price"])/st["entry_price"] if st["position"] == "CALL" else (st["entry_price"] - price)/st["entry_price"]
                if p_pct >= TP_THRESHOLD or (st["position"] == "CALL" and price < st["trailing_sl"]) or (st["position"] == "PUT" and price > st["trailing_sl"]):
                    pnl = (price - st["entry_price"]) if st["position"] == "CALL" else (st["entry_price"] - price)
                    st["stats"]["pnl"] += pnl

                    # Update performance metrics
                    if pnl > 0:
                        st['wins'] += 1
                    else:
                        st['losses'] += 1
                    st['best_trade'] = max(st['best_trade'], pnl)
                    st['worst_trade'] = min(st['worst_trade'], pnl)

                    send_alert(f"EXIT {s} | P&L: {round(pnl,2)}")
                    st["position"] = None; st["entry_price"] = None

            # Entry Logic (Simplified Signal)
            elif ha.iloc[-2]["close"] > ha.iloc[-2]["open"] and ha.iloc[-3]["close"] < ha.iloc[-3]["open"]:
                st["position"], st["entry_price"] = "CALL", price
                st["trailing_sl"] = price - (ind["atr"] * ATR_SL_MULTIPLIER); st["stats"]["trades"] += 1
                send_alert(f"ENTRY CALL {s} at {price}")

        print_dashboard(states)
        time.sleep(30)

start_bot()


  === MULTI-SYMBOL BOT (10:08:09 AM) ===
  SYMBOL       STATUS   POS    ENTRY      PRICE      UNREAL     TRADES  P&L      WINS  LOSSES  WIN%   BEST     WORST   
  ------------------------------------------------------------------------------------------------------------------------------------------------------
  ^NSEI        OPEN     -      0          24154.7    0.0        0       0.0      0     0       0.0    N/A      N/A     
  RELIANCE.NS  OPEN     -      0          1403.6     0.0        0       0.0      0     0       0.0    N/A      N/A     
  HDFCBANK.NS  OPEN     CALL   843.05     841.65     -1.4       0       0.0      0     0       0.0    N/A      N/A     

  Global Trades: 0 | Realized P&L: 0.0 pts | Wins: 0 | Losses: 0 | Win Rate: 0.0% | Best Trade: N/A | Worst Trade: N/A


KeyboardInterrupt: 

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║     TRADING BOT — REVIEWED & FIXED  (Colab Version)         ║
# ╚══════════════════════════════════════════════════════════════╝
#
# CELL 1 — Install (run once)
# !pip install yfinance pandas numpy requests python-dotenv -q

# ════════════════════════════════════════════════════════════════
# CELL 2 — All imports, settings, functions
# ════════════════════════════════════════════════════════════════

import yfinance as yf
import pandas as pd
import numpy as np
import requests
import time
import logging
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from IPython.display import clear_output
from google.colab import userdata
from dotenv import load_dotenv

# ── Secrets ─────────────────────────────────────────────────────
load_dotenv()
TOKEN   = userdata.get("TELEGRAM_TOKEN")
CHAT_ID = userdata.get("CHAT_ID")

# ════════════════════════════════════════════════════════════════
# ✏️  EDIT THESE — everything else auto-adjusts
# ════════════════════════════════════════════════════════════════
SYMBOLS = [
    "^NSEI", "RELIANCE.NS", "HDFCBANK.NS",
]
INTERVAL          = "5m"
MIN_PROBABILITY   = 30    # 0-100. Raise to 60-75 to be more selective
ATR_SL_MULTIPLIER = 1.5   # Wider SL = fewer hits. Try 2.0 if needed
TP_THRESHOLD      = 0.03  # 3% take-profit
# ════════════════════════════════════════════════════════════════

IST = ZoneInfo("Asia/Kolkata")
logging.basicConfig(level=logging.WARNING)


# ──────────────────────────────────────────────────────────────
# BUG FIX 1: detect_profile — added MCX, ^BSE, .BO support
# Original only handled Crypto, .NS, and a generic fallback.
# MCX symbols were falling into US_STOCK profile (wrong timezone,
# wrong market hours, wrong strike rounding — all incorrect).
# ──────────────────────────────────────────────────────────────
def detect_profile(symbol: str) -> dict:
    s = symbol.upper()
    CRYPTO_KW = ["BTC", "ETH", "SOL", "BNB", "XRP", "DOGE", "ADA"]

    if any(k in s for k in CRYPTO_KW) or s.endswith("-USD"):
        return dict(type="CRYPTO",    tz="UTC",
                    hours=None,               doji=0.15, strike=1,  max_daily_loss=3)

    # MCX commodities — Indian exchange, same hours as NSE roughly
    if s.endswith(".MCX"):
        return dict(type="MCX",       tz="Asia/Kolkata",
                    hours=("09:00", "23:30"), doji=0.20, strike=1,  max_daily_loss=3)

    if any(s.startswith(x) for x in ["^NSE", "^BSE"]) or s.endswith((".NS", ".BO")):
        return dict(type="INDIA",     tz="Asia/Kolkata",
                    hours=("09:15", "15:30"), doji=0.20, strike=50, max_daily_loss=3)

    if s in ["^GSPC", "^DJI", "^IXIC", "SPY", "QQQ"]:
        return dict(type="US_INDEX",  tz="America/New_York",
                    hours=("09:30", "16:00"), doji=0.20, strike=5,  max_daily_loss=3)

    return     dict(type="US_STOCK",  tz="America/New_York",
                    hours=("09:30", "16:00"), doji=0.20, strike=1,  max_daily_loss=3)


# ──────────────────────────────────────────────────────────────
# BUG FIX 2: is_market_open — original checked weekday correctly
# but did NOT handle MCX extended hours (open till 11:30 PM IST).
# Now handled via detect_profile hours dict above.
# ──────────────────────────────────────────────────────────────
def is_market_open(profile: dict) -> bool:
    if profile["hours"] is None:
        return True   # Crypto — always open
    tz      = ZoneInfo(profile["tz"])
    now_tz  = datetime.now(tz)
    if now_tz.weekday() >= 5:
        return False  # Saturday=5, Sunday=6
    curr = now_tz.strftime("%H:%M")
    return profile["hours"][0] <= curr <= profile["hours"][1]


# ──────────────────────────────────────────────────────────────
# Telegram
# ──────────────────────────────────────────────────────────────
def ist_now() -> str:
    return datetime.now(IST).strftime("%d-%b-%Y %I:%M:%S %p IST")

def send_alert(msg: str):
    if not TOKEN or not CHAT_ID:
        print(f"[LOG] {msg}")
        return
    try:
        requests.post(
            f"https://api.telegram.org/bot{TOKEN}/sendMessage",
            data={"chat_id": CHAT_ID, "text": msg},
            timeout=5
        )
    except Exception as e:
        print(f"⚠️ Telegram error: {e}")


# ──────────────────────────────────────────────────────────────
# BUG FIX 3: fetch_data — original had no retry.
# A single yfinance timeout crashed the whole symbol for that cycle.
# Added 3-attempt retry with back-off.
# ──────────────────────────────────────────────────────────────
def fetch_data(symbol: str, interval: str, retries: int = 3):
    for attempt in range(1, retries + 1):
        try:
            df = yf.download(symbol, period="2d", interval=interval,
                             auto_adjust=False, progress=False)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = [c[0] for c in df.columns]
            if not df.empty:
                return df
        except Exception as e:
            pass
        time.sleep(3 * attempt)
    return None


# ──────────────────────────────────────────────────────────────
# Heikin Ashi — original logic was correct, kept as-is
# Minor: added explicit float cast to avoid dtype issues
# ──────────────────────────────────────────────────────────────
def heikin_ashi(df: pd.DataFrame):
    ohlc = df[["Open", "High", "Low", "Close"]].apply(
        pd.to_numeric, errors="coerce").dropna()
    if len(ohlc) < 10:
        return None

    o = ohlc["Open"].values.astype(float)
    h = ohlc["High"].values.astype(float)
    l = ohlc["Low"].values.astype(float)
    c = ohlc["Close"].values.astype(float)

    ha_c = (o + h + l + c) / 4
    ha_o = np.zeros_like(o)
    ha_o[0] = (o[0] + c[0]) / 2
    for i in range(1, len(o)):
        ha_o[i] = (ha_o[i - 1] + ha_c[i - 1]) / 2

    return pd.DataFrame({
        "open":  ha_o,
        "close": ha_c,
        "high":  np.maximum.reduce([ha_o, ha_c, h]),
        "low":   np.minimum.reduce([ha_o, ha_c, l]),
    })


# ──────────────────────────────────────────────────────────────
# BUG FIX 4: compute_indicators — original used iloc[-2] for RSI/ATR/EMA.
# But volume_ratio was missing entirely.
# Also: if rolling window had NaN (not enough data), float() would throw.
# Added safe fallback values.
# ──────────────────────────────────────────────────────────────
def compute_indicators(df: pd.DataFrame) -> dict:
    close = pd.to_numeric(df["Close"], errors="coerce")
    high  = pd.to_numeric(df["High"],  errors="coerce")
    low   = pd.to_numeric(df["Low"],   errors="coerce")

    # RSI 14
    delta = close.diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rsi_s = 100 - 100 / (1 + gain / loss)
    rsi   = float(rsi_s.iloc[-2]) if not np.isnan(rsi_s.iloc[-2]) else 50.0

    # ATR 14
    tr  = pd.concat([
        (high - low),
        (high - close.shift(1)).abs(),
        (low  - close.shift(1)).abs()
    ], axis=1).max(axis=1)
    atr_s = tr.rolling(14).mean()
    atr   = float(atr_s.iloc[-2]) if not np.isnan(atr_s.iloc[-2]) else float(tr.mean())

    # EMA 20
    ema_s = close.ewm(span=20, adjust=False).mean()
    ema   = float(ema_s.iloc[-2]) if not np.isnan(ema_s.iloc[-2]) else float(close.iloc[-2])

    # Volume ratio
    vol_ratio = 1.0
    if "Volume" in df.columns:
        vol = pd.to_numeric(df["Volume"], errors="coerce")
        avg = vol.rolling(20).mean().iloc[-2]
        if avg and avg > 0:
            vol_ratio = float(vol.iloc[-2] / avg)

    return {"rsi": rsi, "atr": atr, "ema": ema, "vol_ratio": vol_ratio}


# ──────────────────────────────────────────────────────────────
# BUG FIX 5: Probability scoring — MISSING entirely in original.
# Original just entered on any HA flip with no quality filter.
# This is the #1 reason for excessive SL hits.
# Added full 5-factor scoring (same as single-symbol bot).
# ──────────────────────────────────────────────────────────────
def compute_probability(ha: pd.DataFrame, ind: dict, direction: str) -> tuple:
    score = 0
    breakdown = {}
    curr = ha.iloc[-2]   # last confirmed candle

    # 1. Candle body strength (0-25)
    rng  = curr["high"] - curr["low"]
    body = abs(curr["close"] - curr["open"])
    strength = (body / rng) if rng > 0 else 0
    pts = min(round(strength * 25), 25)
    score += pts
    breakdown["Body Strength"] = f"{pts}/25"

    # 2. RSI zone (0-25)
    rsi = ind["rsi"]
    if direction == "CALL":
        rsi_pts = 25 if 30 <= rsi <= 60 else (20 if rsi < 30 else (10 if rsi <= 70 else 0))
    else:
        rsi_pts = 25 if 40 <= rsi <= 70 else (20 if rsi > 70 else (10 if rsi >= 30 else 0))
    score += rsi_pts
    breakdown["RSI"] = f"{rsi_pts}/25 (RSI={round(rsi,1)})"

    # 3. Volume surge (0-20)
    vr = ind["vol_ratio"]
    vol_pts = 20 if vr >= 2.0 else (15 if vr >= 1.5 else (8 if vr >= 1.0 else 0))
    score += vol_pts
    breakdown["Volume"] = f"{vol_pts}/20 ({round(vr,2)}×)"

    # 4. Prior HA trend — last 5 candles before flip (0-20)
    prior = ha.iloc[-7:-2]
    agree = int((prior["close"] < prior["open"]).sum()) if direction == "CALL" \
       else int((prior["close"] > prior["open"]).sum())
    trend_pts = round((agree / 5) * 20)
    score += trend_pts
    breakdown["Prior Trend"] = f"{trend_pts}/20 ({agree}/5)"

    # 5. Price vs EMA 20 (0-10)
    price   = float(curr["close"])
    ema_pts = 10 if (direction == "CALL" and price > ind["ema"]) or \
                    (direction == "PUT"  and price < ind["ema"]) else 0
    score += ema_pts
    breakdown["EMA Filter"] = f"{ema_pts}/10"

    return score, breakdown


# ──────────────────────────────────────────────────────────────
# BUG FIX 6: check_signal — original entry logic in start_bot()
# was inline and only checked CALL (no PUT signal at all!).
# Original code:
#   elif ha.iloc[-2]["close"] > ha.iloc[-2]["open"] and
#        ha.iloc[-3]["close"] < ha.iloc[-3]["open"]:
#       st["position"] = "CALL"
# — PUT direction was completely missing.
# — No doji filter.
# — Used iloc[-2] as current (live unconfirmed candle).
# Fixed: both directions, doji filter, confirmed candles only.
# ──────────────────────────────────────────────────────────────
def not_doji(row, thresh: float) -> bool:
    rng = row["high"] - row["low"]
    return rng > 0 and abs(row["close"] - row["open"]) > thresh * rng

def check_signal(ha: pd.DataFrame, doji_thresh: float):
    """
    iloc[-3] = prev confirmed candle
    iloc[-2] = current confirmed candle
    iloc[-1] = live forming candle — intentionally IGNORED
    """
    prev = ha.iloc[-3]
    curr = ha.iloc[-2]

    if (prev["close"] < prev["open"] and curr["close"] > curr["open"]
            and not_doji(prev, doji_thresh) and not_doji(curr, doji_thresh)):
        return "CALL"

    if (prev["close"] > prev["open"] and curr["close"] < curr["open"]
            and not_doji(prev, doji_thresh) and not_doji(curr, doji_thresh)):
        return "PUT"

    return None


# ATR-based SL and trailing
def calc_sl(price: float, atr: float, direction: str) -> float:
    return price - atr * ATR_SL_MULTIPLIER if direction == "CALL" \
      else price + atr * ATR_SL_MULTIPLIER

def trail_sl(current_sl: float, price: float, atr: float, direction: str) -> float:
    new_sl = calc_sl(price, atr, direction)
    return max(current_sl, new_sl) if direction == "CALL" \
      else min(current_sl, new_sl)


# ──────────────────────────────────────────────────────────────
# BUG FIX 7: print_dashboard — original printed entry=0 when no
# position (confusing). Also "TRADES" column showed wins+losses
# but stats["trades"] was incremented separately — double-counting.
# Fixed: unified trade counter, cleaner display, IST timestamp.
# ──────────────────────────────────────────────────────────────
def print_dashboard(symbol_states: dict):
    clear_output(wait=True)
    now = datetime.now(IST).strftime("%d-%b-%Y  %I:%M:%S %p  IST")
    print(f"\n  🤖  MULTI-SYMBOL BOT  ·  {now}")
    print(f"\n  {'SYMBOL':<14} {'MKT':<7} {'POS':<5} {'ENTRY':>10} {'PRICE':>10} "
          f"{'UNREAL':>9} {'PROB':>5} {'TRADES':>6} {'W':>4} {'L':>4} "
          f"{'WIN%':>6} {'P&L':>9} {'BEST':>9} {'WORST':>9}")
    print("  " + "─" * 130)

    g = dict(trades=0, wins=0, losses=0, pnl=0.0,
             best=float("-inf"), worst=float("inf"))

    for sym, st in symbol_states.items():
        mkt   = "OPEN" if is_market_open(st["profile"]) else "CLOSED"
        pos   = st["position"] or "—"
        entry = st["entry_price"]
        price = st["latest_price"]
        prob  = st.get("last_prob", "—")

        entry_disp = f"{round(entry,2)}" if entry else "—"

        unreal = 0.0
        if st["position"] == "CALL" and entry:
            unreal = price - entry
        elif st["position"] == "PUT" and entry:
            unreal = entry - price

        w    = st["wins"]
        l    = st["losses"]
        tot  = w + l
        wr   = round(w / tot * 100, 1) if tot > 0 else 0.0
        pnl  = st["pnl"]
        best  = round(st["best"],  2) if st["best"]  != float("-inf") else "—"
        worst = round(st["worst"], 2) if st["worst"] != float("inf")  else "—"

        unreal_str = f"{'📈' if unreal >= 0 else '📉'}{round(unreal,2)}"

        print(f"  {sym:<14} {mkt:<7} {pos:<5} {entry_disp:>10} {round(price,2):>10} "
              f"{unreal_str:>9} {str(prob):>5} {tot:>6} {w:>4} {l:>4} "
              f"{wr:>6} {round(pnl,2):>9} {str(best):>9} {str(worst):>9}")

        g["trades"] += tot
        g["pnl"]    += pnl
        g["wins"]   += w
        g["losses"] += l
        if st["best"]  != float("-inf"): g["best"]  = max(g["best"],  st["best"])
        if st["worst"] != float("inf"):  g["worst"] = min(g["worst"], st["worst"])

    print("  " + "─" * 130)
    gw  = g["wins"]
    gl  = g["losses"]
    gt  = gw + gl
    gwr = round(gw / gt * 100, 1) if gt > 0 else 0.0
    gb  = round(g["best"],  2) if g["best"]  != float("-inf") else "—"
    gwo = round(g["worst"], 2) if g["worst"] != float("inf")  else "—"
    print(f"\n  TOTAL  Trades:{gt}  Wins:{gw}  Losses:{gl}  "
          f"WinRate:{gwr}%  P&L:{round(g['pnl'],2)}  "
          f"Best:{gb}  Worst:{gwo}")
    print(f"\n  Min Probability: {MIN_PROBABILITY}%  |  ATR×{ATR_SL_MULTIPLIER}  |  TP: {int(TP_THRESHOLD*100)}%")
    print("  (Ctrl+C to stop)\n")


# ════════════════════════════════════════════════════════════════
# CELL 3 — Market selector (optional, run before CELL 4)
# ════════════════════════════════════════════════════════════════

def select_markets():
    """
    Run this cell to interactively pick markets.
    Or skip it and edit SYMBOLS directly above.
    """
    options = {
        "1": {"name": "Crypto",    "symbols": ["BTC-USD", "ETH-USD", "SOL-USD"]},
        "2": {"name": "India",     "symbols": ["^NSEI", "RELIANCE.NS", "HDFCBANK.NS"]},
        "3": {"name": "US Stocks", "symbols": ["AAPL", "TSLA", "NVDA", "MSFT"]},
        # BUG FIX 8: MCX was in original SYMBOLS list but not in market_options selector
        # — user had no way to select it interactively. Added here.
        "4": {"name": "MCX Commodities", "symbols": [
            "GOLD.MCX", "SILVER.MCX", "CRUDEOIL.MCX", "COPPER.MCX",
            "NATURALGAS.MCX", "ZINC.MCX"
        ]},
    }

    print("Available Markets:")
    for k, v in options.items():
        print(f"  {k}. {v['name']:20s} → {', '.join(v['symbols'][:3])}{'...' if len(v['symbols'])>3 else ''}")

    # BUG FIX 9: original accepted "India" as text but only split on comma
    # and parsed as int — so typing "India" gave ValueError and fell through
    # to default silently. Now accepts both number and name.
    raw = input("\nEnter numbers (e.g. 1,2) or names (e.g. India,Crypto) or Enter for default (India): ").strip()

    selected = []
    if not raw:
        selected = options["2"]["symbols"]
        print("Defaulting to India.")
    else:
        # Try numeric first, then name match
        parts = [p.strip() for p in raw.replace(",", " ").split()]
        for part in parts:
            if part in options:
                selected.extend(options[part]["symbols"])
            else:
                # name match (case-insensitive)
                for k, v in options.items():
                    if v["name"].lower() == part.lower():
                        selected.extend(v["symbols"])
                        break
                else:
                    print(f"  ⚠️  '{part}' not recognised — skipping")

        if not selected:
            selected = options["2"]["symbols"]
            print("No valid selection. Defaulting to India.")

    result = list(dict.fromkeys(selected))   # deduplicate, preserve order
    print(f"\n✅ Selected: {result}")
    return result

# Uncomment to use interactively:
# SYMBOLS = select_markets()


# ════════════════════════════════════════════════════════════════
# CELL 4 — Main loop
# ════════════════════════════════════════════════════════════════

def start_bot():
    print(f"🚀 Starting bot with {len(SYMBOLS)} symbols: {SYMBOLS}")
    send_alert(
        f"🤖 Multi-Symbol Bot Started\n"
        f"Symbols  : {', '.join(SYMBOLS)}\n"
        f"Min Prob : {MIN_PROBABILITY}%\n"
        f"ATR SL   : {ATR_SL_MULTIPLIER}×\n"
        f"TP       : {int(TP_THRESHOLD*100)}%\n"
        f"Time     : {ist_now()}"
    )

    # Initialise state per symbol
    states = {s: dict(
        position    = None,
        entry_price = None,
        trailing_sl = None,
        latest_price= 0.0,
        profile     = detect_profile(s),
        pnl         = 0.0,
        wins        = 0,
        losses      = 0,
        best        = float("-inf"),
        worst       = float("inf"),
        last_time   = None,
        last_prob   = "—",
        daily_losses= 0,
        last_day    = datetime.now().date(),
    ) for s in SYMBOLS}

    while True:
        for sym in SYMBOLS:
            st = states[sym]

            # Reset daily losses at midnight
            today = datetime.now().date()
            if today != st["last_day"]:
                st["daily_losses"] = 0
                st["last_day"]     = today

            # Daily loss guard
            if st["daily_losses"] >= st["profile"]["max_daily_loss"]:
                continue

            if not is_market_open(st["profile"]):
                continue

            df = fetch_data(sym, INTERVAL)
            if df is None:
                continue

            # Skip duplicate candle
            if df.index[-1] == st["last_time"]:
                continue
            st["last_time"] = df.index[-1]

            ha  = heikin_ashi(df)
            if ha is None:
                continue

            try:
                ind = compute_indicators(df)
            except Exception:
                continue

            price = float(df["Close"].iloc[-1])
            st["latest_price"] = price

            # ── EXIT LOGIC ────────────────────────────────────
            if st["position"]:
                entry = st["entry_price"]
                pos   = st["position"]
                atr   = ind["atr"]

                # Trail the SL
                st["trailing_sl"] = trail_sl(st["trailing_sl"], price, atr, pos)

                # P&L percentage for TP check
                p_pct = (price - entry) / entry if pos == "CALL" \
                   else (entry - price) / entry

                # BUG FIX 10: original TP/SL check was one combined if.
                # This meant if TP was hit it still checked SL on same candle.
                # Now separated into TP first, then SL, with clear exit reason.
                exit_reason = None
                if p_pct >= TP_THRESHOLD:
                    exit_reason = "✅ TP HIT"
                elif pos == "CALL" and price < st["trailing_sl"]:
                    exit_reason = "❌ SL HIT"
                elif pos == "PUT"  and price > st["trailing_sl"]:
                    exit_reason = "❌ SL HIT"

                if exit_reason:
                    pnl = (price - entry) if pos == "CALL" else (entry - price)
                    st["pnl"] += pnl
                    if pnl > 0:
                        st["wins"] += 1
                    else:
                        st["losses"]       += 1
                        st["daily_losses"] += 1
                    st["best"]  = max(st["best"],  pnl)
                    st["worst"] = min(st["worst"], pnl)

                    send_alert(
                        f"{exit_reason}  {sym}\n"
                        f"Direction : {pos}\n"
                        f"Entry     : {round(entry, 2)}\n"
                        f"Exit      : {round(price, 2)}\n"
                        f"P&L       : {round(pnl, 2)} pts\n"
                        f"Time      : {ist_now()}"
                    )
                    st["position"]    = None
                    st["entry_price"] = None
                    st["trailing_sl"] = None

            # ── ENTRY LOGIC ───────────────────────────────────
            elif st["position"] is None:
                signal = check_signal(ha, st["profile"]["doji"])

                if signal:
                    score, breakdown = compute_probability(ha, ind, signal)
                    st["last_prob"]   = f"{score}%"

                    if score >= MIN_PROBABILITY:
                        st["position"]    = signal
                        st["entry_price"] = price
                        st["trailing_sl"] = calc_sl(price, ind["atr"], signal)

                        send_alert(
                            f"📊 ENTRY {signal}  —  {sym}\n"
                            f"Probability : {score}%\n"
                            f"Price       : {round(price, 2)}\n"
                            f"Stop Loss   : {round(st['trailing_sl'], 2)}\n"
                            f"ATR         : {round(ind['atr'], 2)}\n"
                            f"RSI         : {round(ind['rsi'], 1)}\n"
                            f"Time        : {ist_now()}\n"
                            + "\n".join(f"  {k}: {v}" for k, v in breakdown.items())
                        )
                    else:
                        st["last_prob"] = f"{score}%⚠️"

        # Dashboard refresh
        print_dashboard(states)

        # BUG FIX 11: original slept a flat 30s regardless of market.
        # This means on a 5m chart, you might refresh 10 times on the same candle
        # (wasting API calls) or miss a candle. Sleep until next 5m boundary.
        sleep_secs = _seconds_until_next_5min()
        time.sleep(sleep_secs)


def _seconds_until_next_5min() -> float:
    now       = datetime.now()
    delta     = timedelta(minutes=(5 - now.minute % 5))
    next_time = (now + delta).replace(second=5, microsecond=0)
    return max(5.0, (next_time - now).total_seconds())


# ════════════════════════════════════════════════════════════════
# CELL 5 — Run
# ════════════════════════════════════════════════════════════════
start_bot()

🚀 Starting bot with 3 symbols: ['^NSEI', 'RELIANCE.NS', 'HDFCBANK.NS']

  🤖  MULTI-SYMBOL BOT  ·  11-Mar-2026  10:08:45 AM  IST

  SYMBOL         MKT     POS        ENTRY      PRICE    UNREAL  PROB TRADES    W    L   WIN%       P&L      BEST     WORST
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  ^NSEI          OPEN    —              —    24126.2      📈0.0     —      0    0    0    0.0       0.0         —         —
  RELIANCE.NS    OPEN    —              —     1403.9      📈0.0     —      0    0    0    0.0       0.0         —         —
  HDFCBANK.NS    OPEN    CALL       841.1      841.1      📈0.0   46%      0    0    0    0.0       0.0         —         —
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

  TOTAL  Trades:0  Wins:0  Losses:0  WinRate:0.0%  P&L:0.0  Best:—  Worst:—

  Min Probability: 30%  |  ATR×1.5 